# Employee Attrition — Sprint 3
## Feature Engineering, Feature Selection, Hyperparameter Tuning & Final Model

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

## Load Dataset

In [ ]:
df = pd.read_csv("sprint1_encoded.csv")

In [ ]:
df

In [ ]:
df.drop("Unnamed: 0", axis=1, inplace=True)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

## Decode Encoded Boolean Columns

In [ ]:
bool_cols = df.select_dtypes(include=[bool]).columns
df[bool_cols] = df[bool_cols].astype(int)
df.dtypes

## Feature Engineering
Three new features are created from existing signals to increase predictive power.

### Tenure-to-Age Ratio

In [ ]:
df["tenure_age_ratio"] = df["company_tenure"] / (df["age"] + 1)

In [ ]:
df[["company_tenure", "age", "tenure_age_ratio"]].head()

Captures how early in their career an employee joined the company. A low ratio signals a job-hopper pattern; a high ratio indicates loyalty — a strong attrition discriminator.

### Income-per-Promotion

In [ ]:
df["income_per_promotion"] = df["monthly_income_capped"] / (df["number_of_promotions"] + 1)

In [ ]:
df[["monthly_income_capped", "number_of_promotions", "income_per_promotion"]].head()

Employees earning high income relative to promotions received may feel their career growth is stalling, elevating attrition risk regardless of compensation.

### Work Stress Indicator

In [ ]:
df["work_stress"] = df["overtime"] * df["distance_from_home"]

In [ ]:
df[["overtime", "distance_from_home", "work_stress"]].head()

Compounds overtime burden with commute distance. An employee who consistently works overtime while commuting long distances faces compounded stress — a leading attrition driver.

In [ ]:
df.head()

## Separate Features and Target

In [ ]:
x = df.drop("attrition", axis=1)
y = df["attrition"]

In [ ]:
x.shape

In [ ]:
y.shape

## Correlation Analysis

In [ ]:
sns.heatmap(
    x.corr(),
    annot=False,
    cmap="coolwarm"
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = x.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [ ]:
to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.90)
]

print("Columns to drop:", to_drop)

X = x.drop(columns=to_drop)

In [ ]:
X.shape

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Feature Scaling

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [ ]:
x_train_scaled

In [ ]:
x_test_scaled

## Feature Selection — Recursive Feature Elimination (RFE)

In [ ]:
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

In [ ]:
rf_selector

In [ ]:
rfe = RFE(
    estimator=rf_selector,
    n_features_to_select=15
)

In [ ]:
rfe

In [ ]:
rfe.fit(x_train_scaled, y_train)

In [ ]:
rfe.support_

In [ ]:
selected_features = X.columns[rfe.support_]

In [ ]:
selected_features

In [ ]:
x_train_selected = rfe.transform(x_train_scaled)

In [ ]:
x_train_selected

In [ ]:
x_test_selected = rfe.transform(x_test_scaled)

In [ ]:
x_test_selected

## Hyperparameter Tuning — RandomizedSearchCV

In [ ]:
param_dist = {
    "n_estimators"     : [100, 150, 200, 300],
    "max_depth"        : [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
    "max_features"     : ["sqrt", "log2"],
    "bootstrap"        : [True, False]
}

In [ ]:
rand_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [ ]:
rand_search

In [ ]:
rand_search.fit(x_train_selected, y_train)

In [ ]:
rand_search.best_params_

In [ ]:
rand_search.best_score_

## Final Model — Random Forest Classifier

In [ ]:
final_model = rand_search.best_estimator_

In [ ]:
final_model

In [ ]:
y_pred = final_model.predict(x_test_selected)

In [ ]:
y_pred

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
precision = precision_score(y_test, y_pred)
precision

In [ ]:
recall = recall_score(y_test, y_pred)
recall

In [ ]:
f1 = f1_score(y_test, y_pred)
f1

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest (Tuned)")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

## Feature Importance

In [ ]:
importances = final_model.feature_importances_

In [ ]:
importances

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature"   : selected_features,
    "Importance": importances
})

In [ ]:
feature_importance_df

In [ ]:
feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)
feature_importance_df

In [ ]:
sns.barplot(
    x="Importance",
    y="Feature",
    data=feature_importance_df
)
plt.title("Feature Importance — Random Forest Classifier")
plt.tight_layout()
plt.show()

## Cross-Validation Score — Final Model

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    final_model,
    x_train_selected,
    y_train,
    cv=skf,
    scoring="f1"
)

In [ ]:
cv_scores

In [ ]:
print("CV F1 Mean :", round(cv_scores.mean(), 4))
print("CV F1 Std  :", round(cv_scores.std(), 4))

## Sprint 3 — Conclusion

Random Forest Classifier was selected as the final model based on Sprint 2 evaluation where it achieved the highest potential — ensemble power, built-in feature importance, and best overall F1/AUC balance — with hyperparameter tuning applied in Sprint 3 to resolve the overfitting gap seen at baseline. This is confirmed by `best_model.pkl` in the Sprint 4 deployment pipeline (MLflow + Optuna + Streamlit) which stored a tuned Random Forest as the champion.

Three domain-driven features were engineered: **Tenure-to-Age Ratio** (loyalty signal), **Income-per-Promotion** (career progression satisfaction), and **Work Stress Indicator** (overtime x commute burden). RFE with a Random Forest estimator narrowed the feature space to the 15 most informative predictors. RandomizedSearchCV with Stratified 5-Fold cross-validation optimised key hyperparameters — `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, and `bootstrap` — using F1 as the objective metric. The tuned final model is stable across folds (low CV std) and ready for Sprint 4 deployment.